# ♻️ Smart E-Waste Segregation System: YOLOv8 Training & Inference
**Project:** Conveyor-Based Automated Electronic Waste Segregation using YOLOv8 & 6-DOF Robotic Arm  
**GitHub Repository:** [neilgibsonpinto-art/ewaste](https://github.com/neilgibsonpinto-art/ewaste)  
**Target Classes:** `0: pcb`, `1: battery`, `2: display`  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neilgibsonpinto-art/ewaste/blob/main/Smart_EWaste_Colab.ipynb)

## 1. Setup Environment & Clone Project Repository (Foolproof)
Resets the working directory cleanly to avoid `FileExistsError` or `No such file or directory: ewaste`, clones the latest code, and installs YOLOv8.

In [ ]:
import os

# Always reset to root /content directory
%cd /content

# Clean existing clone if re-running
if os.path.exists('/content/ewaste'):
    !rm -rf /content/ewaste

!git clone https://github.com/neilgibsonpinto-art/ewaste.git
%cd /content/ewaste

# Install packages
!pip install -q ultralytics roboflow opencv-python matplotlib Pillow

import torch
import ultralytics
print('PyTorch Version:', torch.__version__)
print('Ultralytics Version:', ultralytics.__version__)
print('GPU Active:', torch.cuda.is_available())

## 2. Dataset Inspection & Preview
Configures `data.yaml` with absolute Colab paths and visualizes sample images.

In [ ]:
import yaml
import glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Configure data.yaml with absolute Colab path
data_yaml_path = Path('/content/ewaste/dataset/data.yaml')
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = '/content/ewaste/dataset'
data_config['train'] = 'images/train'
data_config['val'] = 'images/val'

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print('Dataset YAML Verified:')
print(yaml.dump(data_config))

# Preview training images
train_images = glob.glob('/content/ewaste/dataset/images/train/*.jpg')[:4]
fig, axes = plt.subplots(1, len(train_images), figsize=(16, 4))
if len(train_images) == 1:
    axes = [axes]
for ax, img_p in zip(axes, train_images):
    ax.imshow(Image.open(img_p))
    ax.set_title(Path(img_p).stem)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Train Lightweight YOLOv8-Nano Model
Trains on the 92 e-waste component images using the T4 GPU.

In [ ]:
import os
from ultralytics import YOLO

# Use local weights if present or auto-download
weights_source = '/content/ewaste/yolov8n.pt' if os.path.exists('/content/ewaste/yolov8n.pt') else 'yolov8n.pt'
model = YOLO(weights_source)

# Train model
results = model.train(
    data='/content/ewaste/dataset/data.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/ewaste/runs/train',
    name='ewaste_colab_yolov8n',
    plots=True,
    verbose=True
)

print('Training Complete!')

## 4. Visualize Model Performance Curves
Displays training loss, confusion matrix, and Precision-Recall curves.

In [ ]:
from IPython.display import Image as IPyImage, display
import os

train_dir = '/content/ewaste/runs/train/ewaste_colab_yolov8n'
for img_name in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    p = os.path.join(train_dir, img_name)
    if os.path.exists(p):
        print(f'--- {img_name} ---')
        display(IPyImage(p))

## 5. Test Inference with Object Counter
Runs predictions on test images and displays the segregation tally.

In [ ]:
import glob
from collections import Counter
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Load best weights
trained_model = YOLO('/content/ewaste/runs/train/ewaste_colab_yolov8n/weights/best.pt')

val_images = glob.glob('/content/ewaste/dataset/images/val/*.jpg')[:6]
counter = Counter()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_p in zip(axes.flatten(), val_images):
    res = trained_model.predict(img_p, conf=0.45, verbose=False)[0]
    for box in res.boxes:
        cls_name = res.names[int(box.cls[0])]
        counter[cls_name] += 1
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(Path(img_p).stem)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('=' * 40)
print('  SEGREGATION TALLY SUMMARY')
print('=' * 40)
for k, v in counter.items():
    print(f'  {k.upper()}: {v} item(s)')
print(f'  TOTAL PROCESSED: {sum(counter.values())}')
print('=' * 40)

## 6. Export Model for Edge Devices (Download Weights)
Downloads `best.pt` directly to your laptop for edge deployment.

In [ ]:
from google.colab import files
files.download('/content/ewaste/runs/train/ewaste_colab_yolov8n/weights/best.pt')
print('Downloaded best.pt to your computer!')